# MSMARCO-XI Full-Coverage Production RAG Indexer — Assamese V3 FIXED

## Why V3 exists

The previous Assamese run did **not** fail because of the dataset or indexing logic.

It failed before the notebook started because it tried to install `faiss-cpu` with `pip`, while the Kaggle session could not resolve the package server. That wasted ~220 seconds and produced no output.

This V3 deliberately **does not run `pip install` inside the notebook**.

### REQUIRED Kaggle setup before running

Use Kaggle's Dependency Manager:

**Add-ons → Install Dependencies**

Add exactly:

```text
faiss-cpu==1.15.0
```

Then run the Dependency Manager installation/restart before executing this notebook.

Kaggle documents that its Dependency Manager can install and save external Python dependencies for later notebook executions, avoiding a runtime internet dependency. citeturn555581search2turn555581search5

PyPI currently provides `faiss-cpu` 1.15.0 for CPython 3.10+ on manylinux x86-64, which is compatible with the Python 3.12 Kaggle environment shown in your logs. citeturn555581search0

## Full coverage

This notebook remains full coverage:

- all Assamese training records
- one 384-dimensional multilingual-E5 record vector per source record
- all original passages preserved in compressed Parquet shards
- IVF-PQ FAISS vector database
- multi-strategy chunking functions retained for the second-stage RAG pipeline
- resumable indexing
- no selected-only deletion

## 1. Environment check — NO runtime pip installation

In [ ]:
import importlib.util
import sys

required_modules = [
    "numpy",
    "torch",
    "transformers",
    "huggingface_hub",
    "pyarrow",
    "fsspec",
    "tqdm",
    "faiss",
]

missing = [
    name
    for name in required_modules
    if importlib.util.find_spec(name) is None
]

print("Python:", sys.version.split()[0])

if missing:
    raise RuntimeError(
        "\nMissing required modules: " + ", ".join(missing) +
        "\n\nDO NOT run pip from this notebook.\n"
        "In Kaggle use: Add-ons → Install Dependencies → faiss-cpu==1.15.0\n"
        "Then run the Dependency Manager and restart the notebook session."
    )

import numpy as np
import torch
import faiss

print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("FAISS:", getattr(faiss, "__version__", "installed"))
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU REQUIRED. Kaggle Settings → Accelerator → GPU, then restart."
    )

print("GPU:", torch.cuda.get_device_name(0))
torch.set_float32_matmul_precision("high")

## 2. Hugging Face authentication

In [ ]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

print("HF token available:", bool(HF_TOKEN))
print("Authenticated access is recommended for stable Hub throughput.")

## 3. Fixed Assamese configuration

This notebook is intentionally locked to Assamese so there is no accidental wrong-language run.

In [ ]:
from pathlib import Path

LANGUAGE = "as"
LANGUAGE_NAME = "Assamese"
REPO_FILE = "train/asmtrain.parquet"
PASSAGE_MODE = "translated"

MODEL_NAME = "intfloat/multilingual-e5-small"

# Full corpus coverage.
FULL_RUN = True

# Bounded IVF-PQ training sample.
# This only trains the quantizer; it does NOT reduce indexed coverage.
FAISS_TRAIN_RECORDS = 30_000

FAISS_NLIST = 2048
FAISS_PQ_M = 48
FAISS_PQ_BITS = 8

BATCH_ROWS = 1024
EMBED_BATCH_SIZE = 512
MAX_LENGTH = 384

RECORD_TEXT_MAX_CHARS = 6000
ROWS_PER_RECORD_SHARD = 100_000

ROOT = Path("/kaggle/working/msmarco_xi_full")
LANG_ROOT = ROOT / LANGUAGE
RECORD_ROOT = LANG_ROOT / "records"
RECORD_ROOT.mkdir(parents=True, exist_ok=True)

print("Language:", LANGUAGE_NAME)
print("Source:", REPO_FILE)
print("Full coverage:", FULL_RUN)

## 4. Open the Assamese source Parquet remotely

In [ ]:
from huggingface_hub import hf_hub_url
import fsspec
import pyarrow.parquet as pq

REMOTE_URL = hf_hub_url(
    repo_id="ai4bharat/MSMARCO-XI",
    filename=REPO_FILE,
    repo_type="dataset",
    revision="main",
)

headers = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}

fs = fsspec.filesystem(
    "https",
    headers=headers,
)

remote_handle = fs.open(
    REMOTE_URL,
    "rb",
    block_size=16 * 1024 * 1024,
    cache_type="readahead",
)

parquet_file = pq.ParquetFile(remote_handle)
SOURCE_ROWS = int(parquet_file.metadata.num_rows)

print("Remote file:", REPO_FILE)
print("Source rows:", f"{SOURCE_ROWS:,}")

## 5. GPU embedding model — multilingual-e5-small

In [ ]:
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
).to(device)

model.eval()

@torch.inference_mode()
def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).expand(hidden.size()).float()
    return (hidden * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)

@torch.inference_mode()
def encode_texts(texts, prefix="passage: "):
    if not texts:
        return np.empty((0, model.config.hidden_size), dtype="float32")

    all_vecs = []

    for start in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = [
            prefix + str(x)
            for x in texts[start:start + EMBED_BATCH_SIZE]
        ]

        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        tokens = {
            k: v.to(device, non_blocking=True)
            for k, v in tokens.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            output = model(**tokens)
            embeddings = mean_pool(
                output.last_hidden_state,
                tokens["attention_mask"],
            )
            embeddings = torch.nn.functional.normalize(
                embeddings,
                p=2,
                dim=1,
            )

        all_vecs.append(
            embeddings.float().cpu().numpy().astype("float32")
        )

    return np.vstack(all_vecs)

print("Embedding dimension:", encode_texts(["test"]).shape[1])

## 6. Record representation — one vector per source record

In [ ]:
def build_record_text(record):
    passages = record.get("passages") or {}
    translated = passages.get("Translated_passages") or []
    selected = passages.get("is_selected") or []

    selected_texts = [
        str(translated[i]).strip()
        for i, flag in enumerate(selected)
        if flag == 1
        and i < len(translated)
        and str(translated[i]).strip()
    ]

    if not selected_texts:
        selected_texts = [
            str(x).strip()
            for x in translated[:2]
            if str(x).strip()
        ]

    parts = [
        str(record.get("query", "")).strip(),
        str(record.get("Answer", "")).strip(),
        *selected_texts,
    ]

    return "\n".join(
        part for part in parts if part
    )[:RECORD_TEXT_MAX_CHARS]


def normalize_record(record, local_id):
    passages = record.get("passages") or {}

    return {
        "local_id": int(local_id),
        "query_id": int(record.get("query_id", 0)),
        "query": str(record.get("query", "")),
        "answer": str(record.get("Answer", "")),
        "query_type": str(record.get("query_type", "")),
        "source_lang": str(record.get("source_lang", "")),
        "target_lang": str(record.get("target_lang", "")),
        "english_passages": [
            str(x)
            for x in passages.get("English_passages") or []
        ],
        "translated_passages": [
            str(x)
            for x in passages.get("Translated_passages") or []
        ],
        "is_selected": [
            int(x)
            for x in passages.get("is_selected") or []
        ],
    }

## 7. Train IVF-PQ on a bounded sample
The 30,000-record sample trains the quantizer only. The final index still covers every source record.

In [ ]:
import time

DIM = int(model.config.hidden_size)

train_texts = []
train_start = time.perf_counter()

for batch in parquet_file.iter_batches(
    batch_size=1024,
    columns=["query", "Answer", "passages"],
):
    for record in batch.to_pylist():
        text = build_record_text(record)
        if text:
            train_texts.append(text)

        if len(train_texts) >= FAISS_TRAIN_RECORDS:
            break

    if len(train_texts) >= FAISS_TRAIN_RECORDS:
        break

train_vectors = encode_texts(
    train_texts,
    prefix="passage: ",
)

quantizer = faiss.IndexFlatIP(DIM)

index = faiss.IndexIVFPQ(
    quantizer,
    DIM,
    FAISS_NLIST,
    FAISS_PQ_M,
    FAISS_PQ_BITS,
    faiss.METRIC_INNER_PRODUCT,
)

index.train(train_vectors)

print("IVF-PQ trained")
print("Training records:", len(train_texts))
print("Training time:", round(time.perf_counter() - train_start, 1), "s")
print("Dimension:", DIM)
print("nlist:", FAISS_NLIST)
print("PQ:", f"{FAISS_PQ_M}x{FAISS_PQ_BITS}")

## 8. Compressed complete-record store

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

record_schema = pa.schema([
    ("local_id", pa.int64()),
    ("query_id", pa.int64()),
    ("query", pa.string()),
    ("answer", pa.string()),
    ("query_type", pa.string()),
    ("source_lang", pa.string()),
    ("target_lang", pa.string()),
    ("english_passages", pa.list_(pa.string())),
    ("translated_passages", pa.list_(pa.string())),
    ("is_selected", pa.list_(pa.int8())),
])

def write_record_shard(rows, shard_id):
    if not rows:
        return

    path = RECORD_ROOT / f"records_{shard_id:05d}.parquet"

    table = pa.Table.from_pydict(
        {
            name: [row[name] for row in rows]
            for name in record_schema.names
        },
        schema=record_schema,
    )

    pq.write_table(
        table,
        path,
        compression="zstd",
        compression_level=7,
        use_dictionary=True,
    )

    return path

## 9. Full Assamese indexing — checkpoint after every embedding batch

This loop processes **all source rows**.

Coverage is independent of the 30k IVF-PQ training sample.

In [ ]:
import json
from tqdm.auto import tqdm

INDEX_PATH = LANG_ROOT / "faiss_ivfpq.index"
CHECKPOINT_PATH = LANG_ROOT / "checkpoint.json"
CONFIG_PATH = LANG_ROOT / "config.json"

next_id = 0

# Resume only when both artifacts are present.
if INDEX_PATH.exists() and CHECKPOINT_PATH.exists():
    index = faiss.read_index(str(INDEX_PATH))
    checkpoint = json.loads(
        CHECKPOINT_PATH.read_text(encoding="utf-8")
    )
    next_id = int(checkpoint["next_id"])
    print("Resuming at record:", next_id)

pending_texts = []
pending_ids = []
pending_records = []

shard_rows = []
shard_id = next_id // ROWS_PER_RECORD_SHARD

index_start = time.perf_counter()

def flush_batch():
    global pending_texts
    global pending_ids
    global pending_records
    global shard_rows
    global shard_id
    global next_id

    if not pending_texts:
        return

    vectors = encode_texts(
        pending_texts,
        prefix="passage: ",
    )

    ids = np.asarray(
        pending_ids,
        dtype=np.int64,
    )

    index.add_with_ids(
        vectors,
        ids,
    )

    shard_rows.extend(
        pending_records
    )

    while len(shard_rows) >= ROWS_PER_RECORD_SHARD:
        current = shard_rows[:ROWS_PER_RECORD_SHARD]
        del shard_rows[:ROWS_PER_RECORD_SHARD]

        write_record_shard(
            current,
            shard_id,
        )

        shard_id += 1

    pending_texts.clear()
    pending_ids.clear()
    pending_records.clear()

    next_id = int(ids[-1]) + 1

    faiss.write_index(
        index,
        str(INDEX_PATH),
    )

    CHECKPOINT_PATH.write_text(
        json.dumps(
            {
                "next_id": next_id,
                "vectors_indexed": int(index.ntotal),
                "source_rows": int(SOURCE_ROWS),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

for batch in tqdm(
    parquet_file.iter_batches(
        batch_size=BATCH_ROWS,
        columns=[
            "source_lang",
            "target_lang",
            "Answer",
            "query_id",
            "query_type",
            "passages",
            "query",
        ],
    ),
    desc="FULL ASSAMESE",
):
    for record in batch.to_pylist():

        pending_texts.append(
            build_record_text(record)
        )

        pending_ids.append(
            next_id
        )

        pending_records.append(
            normalize_record(
                record,
                next_id,
            )
        )

        next_id += 1

        if len(pending_texts) >= EMBED_BATCH_SIZE:
            flush_batch()

# Final partial batch.
flush_batch()

# Final partial record shard.
if shard_rows:
    write_record_shard(
        shard_rows,
        shard_id,
    )

faiss.write_index(
    index,
    str(INDEX_PATH),
)

elapsed = time.perf_counter() - index_start

config = {
    "dataset": "ai4bharat/MSMARCO-XI",
    "language": LANGUAGE,
    "language_name": LANGUAGE_NAME,
    "source_file": REPO_FILE,
    "source_rows": int(SOURCE_ROWS),
    "records_indexed": int(index.ntotal),
    "all_records": True,
    "all_passages_preserved": True,
    "embedding_model": MODEL_NAME,
    "embedding_dimension": DIM,
    "index_type": "IVF-PQ",
    "nlist": FAISS_NLIST,
    "pq_m": FAISS_PQ_M,
    "pq_bits": FAISS_PQ_BITS,
    "record_representation": "query + answer + selected/fallback translated passages",
    "elapsed_seconds": elapsed,
}

CONFIG_PATH.write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nFULL INDEX COMPLETE")
print("Source rows:", f"{SOURCE_ROWS:,}")
print("Vectors:", f"{index.ntotal:,}")
print("Elapsed:", round(elapsed, 1), "seconds")
print("FAISS:", INDEX_PATH)
print("Records:", RECORD_ROOT)

## 10. Multi-strategy candidate chunking

These are applied to the passages of retrieved candidate records, not used as four separate full-corpus vector databases.

Strategies:
- fixed-size + overlap
- sentence-aware
- semantic
- metadata-aware

In [ ]:
import re

FIXED_SIZE = 500
FIXED_OVERLAP = 80
SENTENCES_PER_CHUNK = 3
SEMANTIC_THRESHOLD = 0.58

def fixed_chunks(text, size=FIXED_SIZE, overlap=FIXED_OVERLAP):
    text = str(text).strip()
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + size, len(text))
        piece = text[start:end].strip()

        if piece:
            chunks.append(piece)

        if end >= len(text):
            break

        start += size - overlap

    return chunks


def sentence_chunks(text):
    sentences = [
        s.strip()
        for s in re.split(
            r"(?<=[.!?।॥])\s+",
            str(text).strip(),
        )
        if s.strip()
    ]

    return [
        " ".join(
            sentences[i:i + SENTENCES_PER_CHUNK]
        )
        for i in range(
            0,
            len(sentences),
            SENTENCES_PER_CHUNK,
        )
    ]


def metadata_aware_chunks(text, query_type, language):
    return [
        f"[type={query_type} language={language}] {chunk}"
        for chunk in sentence_chunks(text)
    ]


def semantic_chunks(text):
    sentences = [
        s.strip()
        for s in re.split(
            r"(?<=[.!?।॥])\s+",
            str(text).strip(),
        )
        if s.strip()
    ]

    if len(sentences) <= 1:
        return sentences

    vectors = encode_texts(
        sentences,
        prefix="passage: ",
    )

    chunks = []
    current = [sentences[0]]

    for i in range(1, len(sentences)):
        similarity = float(
            np.dot(
                vectors[i - 1],
                vectors[i],
            )
        )

        if similarity < SEMANTIC_THRESHOLD:
            chunks.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])

    if current:
        chunks.append(" ".join(current))

    return chunks

print("Chunking strategies ready: fixed, sentence, semantic, metadata-aware")

## 11. Retrieval benchmark

This measures retrieval only.

It is **not** the final voice-to-answer P50/P70/P100.

In [ ]:
final_index = faiss.read_index(
    str(INDEX_PATH)
)

def retrieve_record_ids(query, top_k=20):
    q_vector = encode_texts(
        [query],
        prefix="query: ",
    )

    scores, ids = final_index.search(
        q_vector,
        top_k,
    )

    return [
        (int(idx), float(score))
        for idx, score in zip(
            ids[0],
            scores[0],
        )
        if idx >= 0
    ]

benchmark_queries = []

for batch in parquet_file.iter_batches(
    batch_size=512,
    columns=["query"],
):
    for record in batch.to_pylist():
        query = str(
            record.get("query", "")
        ).strip()

        if query:
            benchmark_queries.append(
                query
            )

        if len(benchmark_queries) >= 100:
            break

    if len(benchmark_queries) >= 100:
        break

latencies = []

for query in benchmark_queries:
    start = time.perf_counter()
    retrieve_record_ids(
        query,
        top_k=20,
    )
    latencies.append(
        (time.perf_counter() - start) * 1000
    )

latencies = np.asarray(
    latencies,
    dtype=np.float64,
)

print("Benchmark queries:", len(latencies))
print(f"P50: {np.percentile(latencies, 50):.2f} ms")
print(f"P70: {np.percentile(latencies, 70):.2f} ms")
print(f"P100: {np.max(latencies):.2f} ms")

## 12. Final artifact layout

```text
as/
├── faiss_ivfpq.index      ← vector database
├── config.json
├── checkpoint.json
└── records/
    ├── records_00000.parquet
    ├── records_00001.parquet
    └── ...
```

`checkpoint.json` is only for resuming indexing.

`config.json` documents the exact build.

`faiss_ivfpq.index` is the vector retrieval database.

`records/*.parquet` contain the actual source records and passages.